# Historical customer value and baseline feasibility

## 01. Executive overview

A subscription business can use issued invoices to understand historical billing
concentration, cadence and customer relationships. This analysis reconstructs the
accepted fictional Fieldwise Cloud scenario and measures issued GBP visible at each
observation. It also examines the separate monthly-normalised amounts on known active
episodes. Future value and total lifetime economics require separate evidence and
approval. The approved six-month billed-value proxy is evaluated in the companion notebook.

## 02. Business objectives

Which accounts and episodes contribute to observed billed value? How do cadence,
tenure, reactivation and incomplete history affect comparison? The results can inform
questions for billing and account review. All amounts here describe one fictional
population. Misata and UCI remain separate sources.

See the [economic and evaluation record](../../docs/05_ml/phase07_clv_economic_definition_and_validation.md)
for governance and the [replay instructions](../README.md#phase-07-customer-lifetime-value-modelling)
for the temporary source reconstruction.

In [ ]:
from collections import Counter
from decimal import Decimal

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from nowa_customer_intelligence.models.clv.features import customer_features
from nowa_customer_intelligence.models.clv.history import historical_value, issued_concentration
from nowa_customer_intelligence.models.clv.reference import load_reference_history, month_ends
from nowa_customer_intelligence.models.clv.target import TargetBuilder, six_month_end

ledger, inventory = load_reference_history()
cutoff = historical_value(ledger, ledger.cutoff)
month_views = [historical_value(ledger, stamp) for stamp in month_ends(ledger)]
assert cutoff.issues is not None
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.dpi": 120, "axes.titleweight": "bold", "font.size": 10})
blue, green, grey = "#245584", "#148279", "#65778b"


def finish():
    plt.tight_layout()
    plt.show()
    plt.close("all")


def pounds(value):
    return "Unavailable" if pd.isna(value) else f"£{value:,.2f}"


def gbp_axis(axis):
    axis.set_major_formatter(mticker.StrMethodFormatter("£{x:,.0f}"))


display(
    Markdown(
        f"**Source window** {cutoff.history_start:%d %B %Y} to {cutoff.observed_at:%d %B %Y} UTC."
    )
)

## 03. CLV economic definitions

Historical customer value here is gross issued GBP visible in the source window.
A prospective measure needs an event definition, horizon and mature follow-up.
Total CLV is a wider economic measure that may depend on margin, discounting,
terminal value and renewal. The contract includes payment status and dates, but
settlement allocations, refunds, taxes, costs and the recognition policy remain
unavailable. Advance invoices cannot establish recognised revenue or collected cash.

## 04. Customer and subscription architecture

The join path is customer ← canonical episode ← unique invoice. Each invoice enters
one episode and one account total. Snapshot and usage tables are analysed in their
own grains to avoid repeating invoice values. A completed episode retains its
history. A reactivation creates a successor identity under the same account.
Concurrent distinct episodes could both contribute, but the retained reference
supports at most one active episode per account.

In [ ]:
customers = pd.DataFrame(
    [
        {
            "Customer": row.customer_id,
            "Created": pd.Timestamp(row.created_on),
            "Tenure days": row.tenure_days,
            "Episodes": row.established_episodes,
            "Completed": row.completed_episodes,
            "Active": row.active_episodes,
            "Reactivated": row.reactivated_episodes,
            "Issued GBP": row.billing.issued_gbp,
            "Invoices": row.billing.invoice_count,
            "Recency days": row.billing.latest_issue_age_days,
            "Evidence": row.billing.billing_evidence,
        }
        for row in cutoff.customers
    ]
)
episodes = pd.DataFrame(
    [
        {
            "Episode": row.subscription_id,
            "Customer": row.customer_id,
            "Established": pd.Timestamp(row.established_on),
            "Completed": row.completed,
            "Reactivated": row.reactivated,
            "Active": row.active,
            "Issued GBP": row.billing.issued_gbp,
            "Invoices": row.billing.invoice_count,
            "Snapshot GBP per month": row.snapshot_recurring_gbp,
        }
        for row in cutoff.episodes
    ]
)
invoices = pd.DataFrame(
    [
        {
            "Invoice": row.invoice_id,
            "Episode": row.subscription_id,
            "Customer": row.customer_id,
            "Issue": pd.Timestamp(row.issued_on),
            "Available": pd.Timestamp(row.available_at),
            "Cadence": row.cadence.value,
            "Issued GBP": row.issued_gbp,
        }
        for row in cutoff.issues
    ]
)

### Executive financial dashboard

The extraction-cutoff denominators are all visible accounts, canonical episodes and
unique issued invoices. The ordinary median averages the middle pair when the roster
has an even size. The earlier descriptive comparator uses the nearest observed middle
rank. The recurring amount is a monthly-normalised snapshot rate for known active
episodes. The cumulative issued total is a historical flow and uses a different unit basis.

In [ ]:
from statistics import median

total = invoices["Issued GBP"].sum()
known_rates = episodes.loc[episodes["Active"], "Snapshot GBP per month"].dropna()
top_rank = len(customers) // 10
top_share = issued_concentration(cutoff)[top_rank][1] / total
covered = customers["Invoices"].gt(0)
display(
    pd.DataFrame(
        [
            ("Known accounts", f"{len(customers):,}"),
            ("Canonical episodes", f"{len(episodes):,}"),
            ("Unique issued invoices", f"{len(invoices):,}"),
            ("Total issued amount", pounds(total)),
            ("Mean issued amount per account", pounds(total / len(customers))),
            ("Median issued amount per account", pounds(median(customers["Issued GBP"]))),
            (
                "Active / completed episodes",
                f"{episodes['Active'].sum():,} / {episodes['Completed'].sum():,}",
            ),
            ("Accounts with known reactivation", f"{customers['Reactivated'].gt(0).sum():,}"),
            ("Snapshot contractual amount per month", pounds(known_rates.sum())),
            ("Active snapshot coverage", f"{len(known_rates):,} / {episodes['Active'].sum():,}"),
            ("Accounts with at least one visible issue", f"{covered.sum():,} / {len(customers):,}"),
            ("Issued share of the highest-value 10% of accounts", f"{top_share:.2%}"),
        ],
        columns=["Cutoff indicator", "Verified value"],
    )
)

## 05. Financial data inventory

The verified export supplies the source counts. The internal fictional records provide
issue availability and the charged cadence. Account creation dates, establishment
and completion facts establish the temporal ownership context. Historical snapshot
amounts are denominated in GBP per modelled service month, and invoice amounts in
GBP per issue. Usage rows are contextual and do not enter financial totals.

| Source | Grain and evidence |
|---|---|
| Customers | Account identifier and creation date |
| Subscriptions and lifecycle | Episode UUID, account link, timed establishment, activation and termination facts |
| Subscription snapshots | Episode and observation time, known status, cadence and Decimal recurring GBP |
| Invoices and internal billing evidence | Invoice and episode identifiers, issue date, Decimal issued GBP, availability times |
| Monthly usage | Episode and month, separate from issued amounts |

### From source rows to independent financial ledgers

An invoice contributes once via its canonical episode's customer foreign key.
Joining both invoice and monthly snapshot rows directly on episode would multiply
each invoice by the number of matching snapshots. Aggregate each source in its own
grain and join at account and observation only afterwards.

For visible issue set (I_e(t)) on episode (e), let (a_i) be Decimal issued GBP
and (d_i) the UTC issue date. For account (c) with known episode set (E_c(t)),

[
V_e(t)=sum_{iin I_e(t)} a_i,qquad
V_c(t)=sum_{ein E_c(t)}V_e(t),qquad N_c(t)=sum_{ein E_c(t)}|I_e(t)|.
]

Mean issue size is (V_c/N_c) and days since last issue is (t_{mathrm{date}}-max_i d_i).
Both are undefined without an issue. Frequency here is an observed count, with
monthly and annual counts reported separately. For usable active snapshots, the
contractual monthly-normalised amount is the sum of the per-episode monthly rates.
An annual advance charge and its monthly equivalent describe different measures.

[
Y_c(t)=sum_{i in I_c, t<mathrm{issue}(i)le h(t)}a_i
quad	ext{where }h(t)=t+6	ext{ UTC calendar months}
]

This target requires complete issue evidence and availability by ascertainment.
Recognition needs accounting and service-period rules. Cash needs settlement
evidence, margin needs attributable costs, and discounted lifetime value needs an
approved discount rate and future cash-flow basis.

In [ ]:
fig, ax = plt.subplots(figsize=(10.8, 3.8))
ax.set(xlim=(0, 1), ylim=(0, 1), title="Canonical attribution and observation-time separation")
ax.axis("off")
nodes = {
    "Account": (0.12, 0.78),
    "Canonical episode": (0.46, 0.78),
    "Unique invoice": (0.82, 0.78),
    "Lifecycle & snapshots": (0.12, 0.42),
    "Available by observation": (0.46, 0.42),
    "Future issues within horizon": (0.82, 0.42),
    "Historical features": (0.46, 0.10),
    "Mature outcome": (0.82, 0.10),
}
for label, (x, y) in nodes.items():
    ax.text(
        x,
        y,
        label,
        ha="center",
        va="center",
        bbox={"facecolor": "#eef4f9", "edgecolor": blue, "boxstyle": "round,pad=.42"},
    )
for start, end in (
    ("Account", "Canonical episode"),
    ("Canonical episode", "Unique invoice"),
    ("Lifecycle & snapshots", "Available by observation"),
    ("Canonical episode", "Available by observation"),
    ("Unique invoice", "Available by observation"),
    ("Unique invoice", "Future issues within horizon"),
    ("Available by observation", "Historical features"),
    ("Future issues within horizon", "Mature outcome"),
):
    props = {"arrowstyle": "->", "color": grey, "shrinkA": 14, "shrinkB": 18}
    if nodes[start][1] == nodes[end][1]:
        props.update(shrinkA=56, shrinkB=68)
    ax.annotate("", xy=nodes[end], xytext=nodes[start], arrowprops=props)
finish()

In [ ]:
display(pd.DataFrame([{"Dataset": row.dataset, "Verified rows": row.records} for row in inventory]))

## 06. Data quality

Invoice counts and amounts must reconcile independently at issue, episode and
account grain. Availability is bounded by the latest of occurrence, effective and
recorded availability for the matching fictional billing fact. The source date
convention is UTC midnight. The Python roll-up rejects duplicate and orphan
identifiers, inconsistent links and unsupported monetary values. The payment
state does not alter issued GBP.

In [ ]:
total_issued = invoices["Issued GBP"].sum()
assert invoices["Invoice"].is_unique
assert total_issued == customers["Issued GBP"].sum() == episodes["Issued GBP"].sum()
assert len(invoices) == customers["Invoices"].sum() == episodes["Invoices"].sum()
assert set(invoices["Episode"]) <= set(episodes["Episode"])
assert (invoices["Available"] <= cutoff.observed_at).all()
quality = {
    "Duplicate invoice IDs": invoices["Invoice"].duplicated().sum(),
    "Zero-value invoices": (invoices["Issued GBP"] == 0).sum(),
    "Negative issues": (invoices["Issued GBP"] < 0).sum(),
    "Available after UTC issue day": (
        invoices["Available"].dt.tz_convert("UTC").dt.tz_localize(None).dt.normalize()
        > invoices["Issue"]
    ).sum(),
    "Accounts without visible invoices": (customers["Invoices"] == 0).sum(),
}
display(pd.Series(quality, name="Observed records").to_frame())
display(Markdown(f"**Reconciled total issued amount** {pounds(total_issued)}."))

No negative or zero-value invoices appear in this reference. The baseline contract
and scenario restrict the amount to non-negative values. This sample provides no
coverage evidence for credits, refunds, signed reversals or mixed currencies.

## 07. Customer population

Count known accounts and episodes at extraction. The distribution of episodes
describes lifecycle structure. A customer with two episodes can have separated
billing intervals linked by reactivation.

In [ ]:
display(
    pd.Series(
        {
            "Accounts": len(customers),
            "Known episodes": len(episodes),
            "Active episodes": episodes["Active"].sum(),
            "Completed episodes": episodes["Completed"].sum(),
            "Maximum active episodes per account": customers["Active"].max(),
        },
        name="Count",
    ).to_frame()
)
display(customers["Episodes"].value_counts().sort_index().rename("Accounts").to_frame())

## 08. Invoice history

Use issue-date months for the final-cutoff distribution and use as-of month-end
cumulative totals for point-in-time comparisons. The issue-date view conditions on
availability at extraction. Any late-attested charge keeps its original issue
month there. In an earlier as-of view it remains unavailable until attested.

In [ ]:
monthly = (
    invoices.groupby([invoices["Issue"].dt.to_period("M"), "Cadence"])["Issued GBP"]
    .sum()
    .unstack(fill_value=Decimal(0))
)
series = pd.DataFrame(
    {
        "Observation": [view.observed_at for view in month_views],
        "Issued GBP": [
            sum((entry.issued_gbp for entry in view.issues), Decimal(0)) for view in month_views
        ],
        "Accounts": [len(view.customers) for view in month_views],
        "Billed accounts": [
            sum(row.billing.invoice_count > 0 for row in view.customers) for view in month_views
        ],
    }
)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
monthly.astype(float).plot.area(ax=axes[0], color=[green, blue], linewidth=0)
axes[0].set(
    title="Issued amounts by issue month and cadence", xlabel="Issue month", ylabel="GBP issued"
)
axes[1].plot(series["Observation"], series["Issued GBP"].astype(float), color=blue)
axes[1].set(
    title="Cumulative issuance available at month end",
    xlabel="UTC observation",
    ylabel="Cumulative GBP",
)
for ax in axes:
    gbp_axis(ax.yaxis)
    ax.tick_params(axis="x", labelrotation=20)
finish()
display(customers[["Invoices", "Recency days"]].describe().round(2))

Growth in the cumulative curve reflects more observed account history. Advance
annual billing can change monthly totals through timing alone. Invoice recency
is the number of days since the latest visible issue and does not measure
delinquency or customer activity.

## 09. Historical customer value

Show observed amounts across the account roster. Quantiles and the histogram
use the same final-cutoff population. Values are converted to plot coordinates
only after exact Decimal aggregation.

In [ ]:
amounts = customers["Issued GBP"].astype(float)
# Nearest observed quantiles keep the displayed values tied to actual GBP records.
quantiles = customers["Issued GBP"].sort_values().reset_index(drop=True)
display(
    pd.Series(
        {
            f"{q:.0%}": pounds(quantiles.iloc[round(q * (len(quantiles) - 1))])
            for q in (0, 0.25, 0.5, 0.75, 0.9, 0.99, 1)
        },
        name="Observed issued GBP",
    ).to_frame()
)
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(amounts, bins=22, color=blue)
ax.set(title="Historical issued value per account", xlabel="Observed issued GBP", ylabel="Accounts")
gbp_axis(ax.xaxis)
finish()

High values can arise from longer exposure or several advance charges. This
distribution gives no estimate of the future value of those accounts.

## 10. Subscription value

Compare episode totals by known state at the cutoff. Completed episode amounts
remain in the account ledger. Active episode totals remain bounded by the observed
period. Differences reflect exposure, catalogue and cadence as well as lifecycle.

In [ ]:
states = episodes["Completed"].map({True: "Completed", False: "Ongoing"})
comparison = (
    episodes.assign(State=states)
    .groupby("State")
    .agg(Episodes=("Episode", "size"), Invoices=("Invoices", "sum"), Issued=("Issued GBP", "sum"))
)
display(comparison.assign(Issued=comparison["Issued"].map(pounds)))
fig, ax = plt.subplots(figsize=(7, 3.3))
ax.boxplot(
    [episodes.loc[states == name, "Issued GBP"].astype(float) for name in comparison.index],
    tick_labels=comparison.index,
    patch_artist=True,
    boxprops={"facecolor": "#d7e7f1"},
    medianprops={"color": blue},
)
ax.set(
    title="Observed issue totals by episode state",
    xlabel="State at cutoff",
    ylabel="GBP per episode",
)
gbp_axis(ax.yaxis)
finish()

## 11. Billing frequency

Compare cadence at the invoice level. Average invoice size is issued GBP divided
by visible invoice count within cadence. Annual prepay concentrates charges and
cannot be compared with monthly invoice count as if both measured service months.

The examples show actual issues during a fully observed six-month slice. An
annual issue can dominate the window or fall outside it altogether. Monthly
issues usually spread the billed amount across several dates. The plots describe
observed issuance and do not imply renewal, collection or recognised revenue.

In [ ]:
cadence = invoices.groupby("Cadence").agg(
    Invoices=("Invoice", "count"), Accounts=("Customer", "nunique"), Issued=("Issued GBP", "sum")
)
cadence["Average invoice"] = [
    pounds(value / int(count))
    for value, count in zip(cadence["Issued"], cadence["Invoices"], strict=True)
]
display(cadence.assign(Issued=cadence["Issued"].map(pounds)))

anchor = pd.Timestamp("2023-06-30 23:59:59.999999", tz="UTC")
end = pd.Timestamp(six_month_end(anchor.to_pydatetime()))
window = invoices[
    invoices["Issue"].gt(anchor.tz_localize(None)) & invoices["Issue"].le(end.tz_localize(None))
]
fig, axes = plt.subplots(2, 1, figsize=(9, 4.8), sharex=True)
for ax, label, colour in zip(axes, ("monthly", "annual"), (blue, green), strict=True):
    eligible = window[window["Cadence"] == label].sort_values(["Customer", "Issue"])
    example = eligible[eligible["Customer"] == eligible["Customer"].iloc[0]]
    ax.vlines(example["Issue"], 0, example["Issued GBP"].astype(float), color=colour, lw=3)
    ax.axvline(anchor.tz_localize(None), ls=":", color=grey)
    ax.axvline(end.tz_localize(None), ls=":", color=grey)
    ax.set(title=f"{label.title()} · one observed account", ylabel="Issued GBP")
    gbp_axis(ax.yaxis)
axes[-1].set(xlabel="UTC issue date · left boundary excluded, right boundary included")
finish()

### A fully observed annual window without issuance

The initial two schedule examples require a bill inside the window. Add the
third case using an account with only available annual evidence by the anchor
and a confirmed zero from the target builder. Without an issue, zero describes
recorded value in the interval. It says nothing about settlement or inactivity.

In [ ]:
builder = TargetBuilder(ledger, complete_through=ledger.cutoff)
anchor_view = historical_value(ledger, anchor.to_pydatetime())
records = customer_features(anchor_view)
annual_zero = next(
    key
    for key, predictors in records.items()
    if predictors["cadence"] == "annual"
    and builder.at(key, anchor.to_pydatetime(), ascertained_at=ledger.cutoff).future_6m_issued_gbp
    == 0
)
known = invoices.loc[invoices["Customer"].eq(annual_zero)].sort_values("Issue")
fig, ax = plt.subplots(figsize=(8.8, 3.4))
ax.axvspan(
    anchor.tz_localize(None),
    end.tz_localize(None),
    color="#dee9ef",
    label="Six-month target window",
)
ax.vlines(
    known["Issue"],
    0,
    known["Issued GBP"].astype(float),
    color=green,
    lw=3,
    label="Observed issues through extraction",
)
ax.set(
    title="Observed annual account with confirmed zero issuance in the target window",
    xlabel="UTC issue date · one anonymised account",
    ylabel="Issued GBP",
)
gbp_axis(ax.yaxis)
ax.legend()
finish()

## 12. Customer tenure

Account age is elapsed days since the observed creation date. Show its association
with accumulated issued value while keeping the actual denominator visible.
Longer tenure generally offers more chances to observe a bill. The levels reflect
the scenario’s catalogue, schedule and window.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.7))
ax.scatter(customers["Tenure days"], amounts, s=12, alpha=0.4, color=blue)
ax.set(
    title="Observed account tenure and cumulative issued value",
    xlabel="Days since account creation",
    ylabel="Observed issued GBP",
)
gbp_axis(ax.yaxis)
finish()

### Financial distributions and observed exposure

The histogram and boxplot pairs show issue recency, count and historical value
on the same account roster. Tenure bands are fixed at 180 and 365 days. Older
accounts have had more opportunity to receive issues. The bands describe
observed exposure and make no claim about per-month earning rates or segmentation.

In [ ]:
fields = ["Recency days", "Invoices", "Issued GBP"]
financial = customers[fields].astype(float)
display(financial.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.99]).round(2))
fig, axes = plt.subplots(2, 3, figsize=(12, 5.5), gridspec_kw={"height_ratios": [1, 1]})
for idx, field in enumerate(fields):
    sample = financial[field].dropna()
    axes[0, idx].hist(sample, bins=20, color=blue)
    axes[0, idx].set(title=field, ylabel="Accounts")
    axes[1, idx].boxplot(sample, tick_labels=["Accounts"])
    axes[1, idx].set(ylabel=field)
    if field == "Issued GBP":
        gbp_axis(axes[0, idx].xaxis)
        gbp_axis(axes[1, idx].yaxis)
finish()
customers["Observed tenure"] = pd.cut(
    customers["Tenure days"],
    [-1, 179, 364, float("inf")],
    labels=["Under 180 days", "180–364 days", "365 days or more"],
)
exposure = customers.groupby("Observed tenure", observed=True).agg(
    Accounts=("Customer", "size"), Invoices=("Invoices", "sum"), Total_GBP=("Issued GBP", "sum")
)
exposure["GBP per account"] = [
    pounds(v / int(n)) for v, n in zip(exposure["Total_GBP"], exposure["Accounts"], strict=True)
]
display(exposure.assign(Total_GBP=exposure["Total_GBP"].map(pounds)))
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.scatter(customers["Tenure days"], customers["Invoices"], s=12, color=blue, alpha=0.4)
ax.set(
    title="Invoice frequency and observed account age",
    xlabel="Observed tenure, days",
    ylabel="Visible invoices per account",
)
finish()
display(financial.assign(Tenure=customers["Tenure days"]).corr(method="spearman").round(2))

## 13. Issued-value concentration

Rank the observed accounts from highest to lowest issued amount and compare
cumulative sums with the equal-share line. The same pool appears in both
denominators. Concentration measures historical issuance exposure only.

In [ ]:
curve = issued_concentration(cutoff)
ranks, cumulative = zip(*curve, strict=True)
x = np.array(ranks) / len(customers)
y = [float(value / total_issued) for value in cumulative]
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(x, y, color=blue, label="Accounts ranked by issued value")
ax.plot([0, 1], [0, 1], ls="--", color=grey, label="Equal-share reference")
ax.set(
    title="Concentration of observed GBP issuance",
    xlabel="Share of accounts",
    ylabel="Share of issued GBP",
)
for axis in (ax.xaxis, ax.yaxis):
    axis.set_major_formatter(mticker.PercentFormatter(1))
ax.legend()
finish()
display(
    Markdown(
        f"The top 10% account for **{cumulative[len(customers) // 10] / total_issued:.1%}** "
        "of observed issuance."
    )
)

### Account distribution and financial contribution

The empirical distribution gives the share of accounts at or below each observed
amount. The existing concentration line instead gives the share of issued GBP
held by the highest-ranked share of accounts. The table uses equal-count ordered
groups for descriptive deciles. Identical amounts that straddle a boundary may
appear in neighbouring groups. Their ranks convey no economic distinction.

In [ ]:
ordered = customers["Issued GBP"].sort_values(ignore_index=True)
fig, ax = plt.subplots(figsize=(7.8, 3.6))
ax.step(
    ordered.astype(float), np.arange(1, len(ordered) + 1) / len(ordered), where="post", color=blue
)
ax.set(
    title="Empirical distribution of historical account issuance",
    xlabel="Issued GBP per account",
    ylabel="Share of accounts at or below",
)
gbp_axis(ax.xaxis)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1))
finish()
allocation = (np.arange(len(ordered)) * 10 // len(ordered)) + 1
decile = (
    pd.DataFrame({"Decile": allocation, "GBP": ordered})
    .groupby("Decile")["GBP"]
    .agg(Accounts="size", Minimum="min", Maximum="max", Total="sum")
)
share = decile["Total"].map(lambda value: float(value / total))
display(
    decile.assign(
        Minimum=decile["Minimum"].map(pounds),
        Maximum=decile["Maximum"].map(pounds),
        Total=decile["Total"].map(pounds),
        Account_share=decile["Accounts"].map(lambda n: f"{n / len(ordered):.1%}"),
        Issued_share=share.map("{:.1%}".format),
    )
)
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.bar(share.index, share.values, color=blue)
ax.set(
    title="Historical issuance contributed by account-value decile",
    xlabel="Descriptive value decile · lowest to highest",
    ylabel="Share of observed issued GBP",
    xticks=share.index,
)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1))
finish()

## 14. Cohort analysis

Group accounts by creation quarter and compare their as-of mean cumulative issuance
at observation quarter-ends. A dash or empty region denotes no observed cohort at
that date. Older cohorts have greater time to accumulate charges. Cells are
descriptive means in GBP per observed account and should be read with cohort sizes.

In [ ]:
cohorts = []
for view in month_views:
    if view.observed_at.month % 3:
        continue
    month = view.observed_at.strftime("%Y-%m")
    for account in view.customers:
        cohorts.append(
            (str(pd.Period(account.created_on, freq="Q")), month, account.billing.issued_gbp)
        )
cohort_rows = pd.DataFrame(cohorts, columns=["Created quarter", "Observation", "GBP"])
cohort_group = cohort_rows.groupby(["Created quarter", "Observation"])["GBP"].agg(["size", "sum"])
cohort_group["GBP per account"] = [
    value / int(size) for value, size in zip(cohort_group["sum"], cohort_group["size"], strict=True)
]
pivot = cohort_group["GBP per account"].unstack()
fig, ax = plt.subplots(figsize=(10, 4.5))
image = ax.imshow(pivot.astype(float), aspect="auto", cmap="Blues", vmin=0)
ax.set(
    xticks=range(len(pivot.columns)),
    xticklabels=pivot.columns,
    yticks=range(len(pivot.index)),
    yticklabels=pivot.index,
    title="As-of issuance by account creation cohort",
    xlabel="Quarter-end observation (UTC)",
    ylabel="Account creation quarter",
)
ax.tick_params(axis="x", labelrotation=45)
ax.grid(False)
fig.colorbar(image, ax=ax, label="Mean cumulative GBP per observed account")
finish()
display(cohort_group["size"].unstack().fillna("—"))

Growing cohort totals reflect repeated measurement of the same accounts and time
in the observation window. The heatmap does not provide independent validation
samples or prospective effects.

## 15. Reactivation analysis

Compare customers with and without a known reactivated episode by the final cutoff.
Keep the prior and successor invoice histories distinct. Final reactivation status
is retrospective for this comparison and cannot be used as an earlier feature.

The timeline retains two canonical episodes under one account. Issuance marks
remain assigned to the episode that produced them. The example illustrates
lineage under the generator and gives no estimate of commercial reactivation.

In [ ]:
reactivation = (
    customers["Reactivated"].gt(0).map({False: "No known reactivation", True: "Reactivated"})
)
summary = (
    customers.assign(Group=reactivation)
    .groupby("Group")
    .agg(Accounts=("Customer", "size"), Episodes=("Episodes", "sum"), Issued=("Issued GBP", "sum"))
)
display(
    summary.assign(
        Mean=[
            pounds(v / int(n)) for v, n in zip(summary["Issued"], summary["Accounts"], strict=True)
        ],
        Issued=summary["Issued"].map(pounds),
    )
)

example_id = sorted(customers.loc[customers["Reactivated"].gt(0), "Customer"])[0]
lineage = sorted(
    (row for row in ledger.episodes if row.episode.customer_id == example_id),
    key=lambda row: row.episode.start_date,
)
fig, ax = plt.subplots(figsize=(9, 2.8))
for index, row in enumerate(lineage):
    left = pd.Timestamp(row.episode.start_date)
    right = pd.Timestamp(row.episode.end_date or cutoff.observed_at.date())
    charges = invoices[invoices["Episode"] == row.episode.subscription_id]
    ax.plot([left, right], [index, index], lw=7, color=[blue, green][index])
    ax.scatter(charges["Issue"], [index] * len(charges), marker="|", color="#20262d", zorder=3)
ax.set(
    title="Observed predecessor and successor · one reactivated account",
    xlabel="UTC source date · vertical marks are issues",
    yticks=[0, 1],
    yticklabels=["Predecessor", "Successor"],
)
ax.invert_yaxis()
finish()

Reactivated accounts in Fieldwise follow a deterministic scenario rule. Historical
differences may reflect their longer or differently timed exposure. They do not
estimate the effect of a retention activity.

## 16. Recurring contractual amount

Use the latest usable snapshot for each known active episode at cutoff. This
monthly-normalised stock measure remains separate from the issued-flow ledger.
An annual charge can cover several future service months, while the normalised
amount represents a monthly equivalent. Neither amount establishes recognition.

In [ ]:
active = episodes[episodes["Active"]]
known = active["Snapshot GBP per month"].dropna()
display(
    pd.Series(
        {
            "Known active episodes": len(active),
            "With usable recurring amount": len(known),
            "Missing recurring amount": len(active) - len(known),
            "Snapshot total GBP per month": pounds(known.sum()),
        },
        name="Cutoff evidence",
    ).to_frame()
)
display(known.value_counts().sort_index().rename("Active episodes").to_frame())

## 17. Historical financial coverage

A present feed can give no visible invoice for an account at an early landmark.
That observed zero differs from an unavailable feed or a literal zero invoice.
The coverage chart uses created accounts as the denominator at each month end.
Accounts before creation are outside that denominator. Even full reference
coverage at extraction cannot establish completeness before April 2021 or after
March 2024, or establish production billing completeness.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(series["Observation"], series["Billed accounts"] / series["Accounts"], color=green)
ax.set(
    title="Month-end account coverage by visible invoices",
    xlabel="UTC observation",
    ylabel="Share of created accounts with a visible issue",
    ylim=(0, 1.04),
)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1))
finish()
display(pd.Series(Counter(customers["Evidence"]), name="Cutoff accounts").to_frame())

## 18. Baseline CLV methods

Keep the descriptive historical comparators separate from the fitted research benchmarks. Compare the
observed mean and the nearest observed median at the common cutoff. A future
constant prediction using a training median, cohort benchmark or contractual
schedule must use mature outcomes and a training-only fitting population.

## 19. Baseline assumptions

The mean divides total issuance by the count of observed accounts at cutoff.
The median summarises a typical historical total under the observed window.
The snapshot amount is a contractual rate. Each answers a different question
and uses a different denominator. None assumes renewals, discounting or margin.

In [ ]:
display(
    pd.DataFrame(
        [
            (
                "Observed mean issued GBP",
                pounds(total_issued / len(customers)),
                "Known accounts at cutoff",
            ),
            (
                "Nearest observed median",
                pounds(quantiles.iloc[round((len(quantiles) - 1) / 2)]),
                "Nearest observed account value at the median rank",
            ),
            (
                "Snapshot recurring total",
                pounds(known.sum()),
                "GBP per month across usable active snapshots",
            ),
        ],
        columns=["Descriptive comparator", "Value", "Basis"],
    )
)

## 20. Sensitivity to observation headroom

Compare calendar headroom at three illustrative horizons. Count month-end
landmarks whose shifted date and time remain inside the extraction window.
An observation with calendar room may still lack complete financial follow-up.
No financial outcome, forecast or discount sensitivity is calculated.

In [ ]:
dates = pd.DatetimeIndex(series["Observation"])
screen = pd.DataFrame(
    [
        {
            "Horizon (calendar months)": months,
            "Landmarks with calendar room": int(
                ((dates + pd.offsets.MonthEnd(months)) <= ledger.cutoff).sum()
            ),
            "Observed landmarks": len(dates),
        }
        for months in (3, 6, 12)
    ]
)
display(screen)
fig, ax = plt.subplots(figsize=(6.8, 3.3))
ax.bar(
    screen["Horizon (calendar months)"].astype(str),
    screen["Landmarks with calendar room"],
    color=blue,
)
ax.set(
    title="Calendar headroom by illustrative horizon",
    xlabel="Calendar months",
    ylabel="Month-end observations",
    ylim=(0, len(dates)),
)
finish()

### Approved six-month outcome and baselines

`future_6m_issued_gbp` is gross issued GBP across all legitimate episodes of an eligible
account in (observation, six-month end]. For example, 31 August 2023 maps to
29 February 2024 at the same UTC time. The issue date uses the UTC midnight
convention. Reactivations can contribute future issues without entering earlier
features. Full fictional coverage and invoice availability support mature zeros.
Immature or incomplete observations carry unknown outcomes. Neither cash nor
profit is inferred.

The maturity chart counts customer observations, including repeated accounts,
across the reference month-end roster before split selection. The fitted
training-median comparison uses only the locked validation partition.
Annual billing produces uneven target values because a six-month horizon may
contain a full advance bill or none. A schedule projection would require an
approved retention or renewal assumption, so the historical recurring rate
remains a descriptive comparator.

The six-month timeline above uses the same UTC date arithmetic as the approved
target. Its vertical boundary marks apply to the issue date. A target also
requires complete follow-up and available evidence by ascertainment. The maturity
chart counts repeated customer observations, including those outside the fitted
population.

In [ ]:
import os
from pathlib import Path

from nowa_customer_intelligence.models.clv.design import assemble
from nowa_customer_intelligence.models.clv.research import WORKSPACE, load_report

project = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
workspace = Path(os.environ.get("NCI_CLV_WORKSPACE", project / WORKSPACE))
report = load_report(workspace) if (workspace / "results" / "manifest.json").is_file() else None
if report is None:
    _, audit = assemble(ledger, complete_through=ledger.cutoff)
else:
    audit = report["design"]["development_audit"]
state = pd.Series(audit["states"]).rename(
    {
        "mature_positive": "Mature positive",
        "mature_zero": "Mature zero",
        "immature": "Immature",
    }
)
display(state.rename("Customer observations").to_frame())
state.plot.barh(
    color=blue,
    title="Six-month billed-value maturity in the reference roster",
    xlabel="Customer month-end observations",
    ylabel="",
)
finish()
if report is not None:
    display(
        pd.Series(report["scores"]["Training median"]["validation"]).rename("Validation").to_frame()
    )
else:
    display(Markdown("The fitted baseline comparison requires the verified local research report."))

### Worked trace of a mature six-month outcome

Choose one account whose successor is established inside the worked window and
whose verified outcome includes an invoice from that successor. The data shown
below omit account and canonical keys. Episode letters preserve the fact that
two distinct canonical episodes contribute. Dates and amounts come from the
observed ledger. The target amount and maturity status come from the target builder.

### Maturity over the reference observation calendar

The outcome counts are customer observations at each month end. Different months
have different roster sizes, and accounts can repeat. An immature target has
insufficient future calendar coverage. An unavailable or unresolved outcome would
remain unknown even with adequate calendar headroom.

In [ ]:
example_time = pd.Timestamp("2022-12-31 23:59:59.999999", tz="UTC").to_pydatetime()
known_view = historical_value(ledger, example_time)
example_features = customer_features(known_view)
horizon = six_month_end(example_time)
future_start = [
    row
    for row in ledger.episodes
    if row.prior_subscription_id
    and row.episode.customer_id in example_features
    and example_time.date() < row.episode.start_date <= horizon.date()
]
issue_groups = invoices.groupby("Episode")
successor = next(
    row
    for row in sorted(future_start, key=lambda row: row.episode.customer_id)
    if issue_groups.get_group(row.episode.subscription_id)["Issue"]
    .le(pd.Timestamp(horizon.date()))
    .any()
)
key = successor.episode.customer_id
target = builder.at(key, example_time, ascertained_at=ledger.cutoff)
aliases = {
    successor.prior_subscription_id: "Predecessor",
    successor.episode.subscription_id: "Successor",
}
history_evidence = [row for row in known_view.issues if row.customer_id == key]
display(
    pd.DataFrame(
        [
            ("Account", "Worked account A"),
            ("Observation · exclusive", target.observed_at.isoformat()),
            ("Horizon end · inclusive", target.horizon_end.isoformat()),
            ("Complete fictional ledger through", ledger.cutoff.isoformat()),
            ("Maturity", target.state.value),
            ("Known episodes at observation", example_features[key]["episodes"]),
            ("Available historical issues", len(history_evidence)),
            ("Historical billed value", pounds(example_features[key]["issued_gbp"])),
            ("Known active episodes at observation", example_features[key]["active"]),
            ("Realised six-month billed value", pounds(target.future_6m_issued_gbp)),
        ],
        columns=["Evidence", "Value"],
    )
)
history_table = pd.DataFrame(
    [
        (
            aliases[row.subscription_id],
            row.issued_on,
            row.available_at.isoformat(),
            pounds(row.issued_gbp),
        )
        for row in history_evidence
    ],
    columns=["Episode", "Issue date", "Available UTC", "Issued GBP"],
)
display(history_table)
future_listing = invoices.loc[
    invoices["Customer"].eq(key)
    & invoices["Issue"].gt(pd.Timestamp(example_time).tz_localize(None))
    & invoices["Issue"].le(pd.Timestamp(target.horizon_end).tz_localize(None))
].copy()
future_listing["Episode"] = future_listing["Episode"].map(aliases)
assert future_listing["Issued GBP"].sum() == target.future_6m_issued_gbp
display(
    future_listing[["Episode", "Issue", "Available", "Issued GBP"]].assign(
        **{"Issued GBP": future_listing["Issued GBP"].map(pounds)}
    )
)

In [ ]:
maturity_rows = []
for view in month_views:
    states = Counter(
        builder.at(account.customer_id, view.observed_at, ascertained_at=ledger.cutoff).state.value
        for account in view.customers
    )
    maturity_rows.append({"Month end": view.observed_at.strftime("%Y-%m"), **states})
period = pd.DataFrame(maturity_rows).set_index("Month end").fillna(0).astype(int)
counts = period.sum()
display(
    pd.DataFrame(
        {
            "Observations": counts,
            "Share of observation roster": (counts / counts.sum()).map("{:.1%}".format),
        }
    )
)
fig, ax = plt.subplots(figsize=(9, 3.7))
for column, colour in zip(period.columns, (blue, green, grey), strict=False):
    ax.plot(
        period.index,
        period[column] / period.sum(axis=1),
        color=colour,
        label=column.replace("_", " ").title(),
    )
ax.set(
    title="Ascertained six-month target coverage by observation month",
    xlabel="UTC month end",
    ylabel="Share of customer observations",
)
ax.set_xticks(range(0, len(period), 6), period.index[::6])
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1))
ax.legend()
finish()

The worked successor is visible in the mature ledger. It has no role in predictors
at the earlier observation. The temporal screen describes recorded gross issuance.
The training median benchmark minimises the sum of absolute deviations because
the number of targets to either side is balanced at a median. A flat forecast is
transparent but cannot model issue timing. Contractual projections remain deferred
until explicit retention, renewal and charge rules can be supported.

The screen reduces the available landmarks as the horizon grows. It gives an upper
bound before outcome lag, account eligibility, chronology and group isolation.
These horizons remain discussion options.

## 21. Business interpretation

Billed value concentration can help frame account exposure and billing-review
questions. Cadence can shift issue amounts without an equivalent change in
monthly contractual value. The number of linked episodes gives context to
historical balances. Commercial decisions need accounting, completeness and
customer evidence beyond this synthetic scope.

## 22. Conclusions

The historical pipeline reconciles one observed GBP amount across invoices,
episodes and accounts while preserving temporal availability and reactivation
identity. The companion notebook evaluates the approved six-month proxy under the locked
evidence and evaluation rules. The
Phase 06 reserve and churn probabilities were outside this analysis.